## 1 · How an LLM makes a decision today

The search router (`rag_pipeline/router_llm.py` in iguide-ai) sends a prompt like this to an LLM (gpt-5.6-luna):

```text
Decide which search methods to add for this query.
Query: most popular notebooks
Guidelines:
- spatial: enable when locations/coordinates/regions/distance/routing/maps are involved ...
- graph: enable when entity-relationship traversal ... helps (authors, organizations, tags, related resources)
Return STRICT JSON only: {"use_semantic": true|false, "use_spatial": true|false, "use_graph": true|false, "rationale": {...}}
```

The LLM **writes** a reply one token at a time, for example `{"use_semantic": false, "use_spatial": false, "use_graph": false, "rationale": {...}}`. Code then parses the JSON and reads off the two flags. That works, but:

- **The answer is free text.** The code has to hope it parses, and has to fall back to something when it doesn't.
- **There is no confidence.** You get `true` or `false`, never "probably".
- **It's slow and costs money.** About 1.6 s per decision over the API in our tests, and paid per call.
- **It can only be steered through the prompt.** This prompt never mentions popularity, so the LLM sends *most popular notebooks* to no graph search, even though the knowledge graph is where popularity lives.

## 2 · How a decision model makes a decision

A decision model never writes text. You give it:

- **a state:** the thing to judge, such as a search query;
- **questions** with a fixed set of answers.

It returns **a probability for every answer**, in a single pass through the network. Laya supports three question types:

| type | answers | returns |
|---|---|---|
| `noul` | yes / no | P(yes) |
| `choice` | one of K options | a probability for each option |
| `score` | ordered levels 0..K-1 | the expected level |

The router's decision becomes two yes/no questions:

In [1]:
import sys
sys.modules["torchvision"] = None   # Laya never uses torchvision, and the hub's copy doesn't match the newer torch

In [1]:
import sys
# The hub's geoai environment ships torch add-ons compiled for its own, older torch. Laya uses none of them,
# so hide them from transformers. Must run first in a fresh kernel.
BLOCK = ["torchvision", "torchaudio", "torchcodec", "torchao", "torchtext",
         "xformers", "flash_attn", "bitsandbytes", "deepspeed", "apex"]
loaded = [m for m in ["transformers", *BLOCK] if m in sys.modules]
assert not loaded, f"Restart the kernel first, then run this cell before any other cell (already loaded: {loaded})"
for m in BLOCK:
    sys.modules[m] = None

In [2]:
# Replace Git LFS pointer files with the real weights, downloaded from GitHub and checked against the
# SHA-256 in each pointer. Run it from the repo folder; it skips files that are already real.
import hashlib, urllib.request
from pathlib import Path

REPO = "YunfanKang/laya-decision-models"
for name in ("laya-vanilla", "laya-finetuned"):
    f = Path("checkpoints") / name / "model.safetensors"
    if not f.read_bytes()[:40].startswith(b"version https://git-lfs"):
        print(f"{f}: already the real weights ({f.stat().st_size / 1e6:.0f} MB)")
        continue
    pointer = dict(line.split(" ", 1) for line in f.read_text().splitlines() if " " in line)
    oid, size = pointer["oid"].split(":", 1)[1], int(pointer["size"])
    url = f"https://media.githubusercontent.com/media/{REPO}/main/{f.as_posix()}"
    print(f"downloading {f} ({size / 1e6:.0f} MB) ...", flush=True)
    tmp, sha, done = f.with_name(f.name + ".part"), hashlib.sha256(), 0
    with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as out:
        while chunk := r.read(1 << 20):
            sha.update(chunk)
            out.write(chunk)
            done += len(chunk)
            if done % (100 << 20) < (1 << 20):
                print(f"  {done / size:4.0%}", flush=True)
    if sha.hexdigest() != oid or done != size:
        tmp.unlink()
        raise RuntimeError(f"{f}: download incomplete or corrupted; re-run this cell")
    tmp.replace(f)
    print(f"  done, checksum verified")

checkpoints/laya-vanilla/model.safetensors: already the real weights (843 MB)
checkpoints/laya-finetuned/model.safetensors: already the real weights (843 MB)


In [3]:
# Ask vanilla Laya your own question: type a query, then the question, then the choices one per line.
import warnings
from pathlib import Path

import torch
from laya.agent import Agent

if "vanilla_model" not in globals():                     # load once; re-running the cell reuses it
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        vanilla_model = Agent("checkpoints/laya-vanilla", device="cuda" if torch.cuda.is_available() else "cpu")

while True:
    query = input("Query (press Enter on an empty line to stop): ").strip()
    if not query:
        break
    instructions = input("Question: ").strip() or "Which option best fits this query?"
    print("Choices, one per line, as 'label' or 'label: description'. Empty line when done.")
    choices = {}
    while True:
        line = input(f"  choice {len(choices) + 1}: ").strip()
        if not line:
            if len(choices) >= 2:
                break
            print("  enter at least two choices")
            continue
        label, _, description = line.partition(":")
        choices[label.strip()] = description.strip() or None

    budget = max(192, 24 * len(choices) + 64)           # long lists of choices need a bigger option budget
    answer = vanilla_model.system_one(
        query, {"q": {"type": "choice", "instructions": instructions, "criteria": choices}},
        max_len=budget + 320, head_max_len=budget)["answers"]["q"]

    print(f"\nQuery:    {query}\nQuestion: {instructions}")
    for label, p in sorted(answer["probabilities"].items(), key=lambda kv: -kv[1]):
        print(f"  {p:6.1%}  {'█' * round(p * 40):<40}  {label}")
    print(f"→ {answer['choice']}\n")

Query (press Enter on an empty line to stop):  I'm a human
Question:  I am hungry, where should I go


Choices, one per line, as 'label' or 'label: description'. Empty line when done.


  choice 1:  Bathroom
  choice 2:  Bedroom
  choice 3:  Kitchen
  choice 4:  



Query:    I'm a human
Question: I am hungry, where should I go
   46.0%  ██████████████████                        Kitchen
   27.3%  ███████████                               Bedroom
   26.8%  ███████████                               Bathroom
→ Kitchen



KeyboardInterrupt: Interrupted by user

# Run vanilla and fine-tuned Laya on the test sets

This notebook loads the two checkpoints shipped in `checkpoints/` and asks both of them every question in the hand-labelled test sets:

- **Search routing:** 96 catalog queries. Does each one need the graph search? The spatial search?
- **First tool:** 42 held-out analysis requests. Which of 53 tools should run first?

It then shows the accuracy, every individual answer, and where the two models disagree. `laya_decision_models.ipynb` explains what a decision model is; this notebook only runs the models.

**Runtime:** about 1 minute on a GPU. On a 2-CPU machine it takes about 20 minutes, almost all of it the 53-option tool questions. Set `TOOL_LIMIT` below to run fewer.

In [4]:
import json, time, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from pathlib import Path

import pandas as pd
import torch
from laya.agent import Agent

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 70)

CHECKPOINTS = {"vanilla": Path("checkpoints/laya-vanilla"), "fine-tuned": Path("checkpoints/laya-finetuned")}
for d in CHECKPOINTS.values():
    w = d / "model.safetensors"
    assert w.exists() and w.stat().st_size > 10_000_000, f"{w} is missing or a Git LFS pointer: run `git lfs pull`"

QUESTIONS = json.loads(Path("data/questions.json").read_text(encoding="utf-8"))
# Each model is asked the router questions in the wording that suits it: the fine-tuned model the exact wording
# it was trained on, vanilla Laya the wording that scored best for it zero-shot.
ROUTER_WORDING = {"vanilla": QUESTIONS["router_zero_shot"], "fine-tuned": QUESTIONS["router_finetuned"]}
TOOL_Q = {"tool": QUESTIONS["tool"]}

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
ROUTER_SET = read("data/eval/router_eval_queries.jsonl")
TOOL_SET = read("data/eval/tool_eval_holdout.jsonl")
TOOL_LIMIT = None          # e.g. 10 for a quick run on CPU

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE} · {len(ROUTER_SET)} routing queries · {len(TOOL_SET[:TOOL_LIMIT])} tool requests")
print("\nrouter questions, as each model is asked them:")
for name, q in ROUTER_WORDING.items():
    print(f"  {name:10} graph:   {q['graph']['instructions']}\n  {'':10} spatial: {q['spatial']['instructions']}")

device: cpu · 96 routing queries · 42 tool requests

router questions, as each model is asked them:
  vanilla    graph:   Does this query name a specific person, organization or tag?
             spatial: Does this query mention a place or geographic location?
  fine-tuned graph:   Does this query name a specific person, organization or tag, ask what is popular, or ask for resources linked to another resource?
             spatial: Does this query mention a specific place, coordinates, or a distance or area on a map?


## Call the models

One model at a time is loaded, so peak memory stays at about 2.5 GB on CPU. Each routing query is one forward pass that answers both yes/no questions. Each tool request is one forward pass over all 53 options.

In [6]:
# Run both models on a few examples from the test sets instead of the full sets.
# The result cells below keep working: they read ROUTER_SET, TOOL_SET and results.
import os

EXAMPLE_QUERIES = ["most popular notebooks", "datasets maintained by NOAA", "publications by John Smith",
                   "weather stations near Chicago", "Sentinel-2 imagery over Nairobi",
                   "NOAA precipitation datasets near Houston", "how do I compute NDVI from Sentinel-2 bands?",
                   "machine learning for flood prediction", "tutorials for GeoPandas"]
EXAMPLE_REQUESTS = ["Pull the boundary of Travis County, Texas.", "Grid the permits into 1 km squares and shade by count.",
                    "Look for datasets on wildfire smoke exposure.", "What does Moran's I measure?",
                    "How alike are central Paris and central Berlin from orbit?"]
ROUTER_SET = [r for q in EXAMPLE_QUERIES for r in read("data/eval/router_eval_queries.jsonl") if r["query"] == q]
TOOL_SET = [r for q in EXAMPLE_REQUESTS for r in read("data/eval/tool_eval_holdout.jsonl") if r["request"] == q]
TOOL_LIMIT = None

if DEVICE == "cpu":   # torch counts the host's cores; on a shared hub that oversubscribes your CPU share
    try:
        q, period = open("/sys/fs/cgroup/cpu.max").read().split()
        threads = os.cpu_count() if q == "max" else max(1, int(q) // int(period))
    except (OSError, ValueError):
        threads = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
    torch.set_num_threads(threads)
print(f"device {DEVICE} · torch threads {torch.get_num_threads()} · {len(ROUTER_SET)} queries · {len(TOOL_SET)} requests")


def run(name):
    print(f"{name}: loading ...", flush=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = Agent(str(CHECKPOINTS[name]), device=DEVICE)
    router = model.predict_batch([r["query"] for r in ROUTER_SET], ROUTER_WORDING[name])
    print(f"  routing: {len(router)} queries done", flush=True)
    tools = []
    for i, r in enumerate(TOOL_SET, 1):
        tools.append(model.system_one(f"{r['context']}\nRequest: {r['request']}", TOOL_Q, max_len=1600, head_max_len=1400))
        print(f"  tools: {i}/{len(TOOL_SET)}", flush=True)
    return {"router": [{f: a["answers"][f]["noul"] for f in ("graph", "spatial")} for a in router],
            "tools": [a["answers"]["tool"]["probabilities"] for a in tools]}


results = {name: run(name) for name in CHECKPOINTS}

device cpu · torch threads 8 · 9 queries · 5 requests
vanilla: loading ...
  routing: 9 queries done
  tools: 1/5
  tools: 2/5
  tools: 3/5
  tools: 4/5
  tools: 5/5
fine-tuned: loading ...
  routing: 9 queries done
  tools: 1/5
  tools: 2/5
  tools: 3/5
  tools: 4/5
  tools: 5/5


## Search routing: accuracy

Scored on the 87 queries not marked ambiguous, with a cut-off of 0.5.

In [7]:
clear = [i for i, r in enumerate(ROUTER_SET) if not r.get("ambiguous")]
acc = {}
for name, res in results.items():
    pred = [{f: res["router"][i][f] >= 0.5 for f in ("graph", "spatial")} for i in range(len(ROUTER_SET))]
    acc[name] = {"graph accuracy": sum(pred[i]["graph"] == ROUTER_SET[i]["graph"] for i in clear) / len(clear),
                 "spatial accuracy": sum(pred[i]["spatial"] == ROUTER_SET[i]["spatial"] for i in clear) / len(clear),
                 "both right": sum(pred[i]["graph"] == ROUTER_SET[i]["graph"] and pred[i]["spatial"] == ROUTER_SET[i]["spatial"]
                                   for i in clear) / len(clear)}
pd.DataFrame(acc).T.round(3)

,graph accuracy,spatial accuracy,both right
vanilla,0.556,0.778,0.444
fine-tuned,0.889,1.000,0.889


## Search routing: every answer

Each cell shows P(yes); ✗ marks a disagreement with the gold label. Rows marked *amb.* are ambiguous and not scored.

In [8]:
def cell(p, gold):
    return f"{p:.2f}" + ("" if (p >= 0.5) == gold else " ✗")


rows = []
for i, r in enumerate(ROUTER_SET):
    row = {"query": r["query"][:70] + (" (amb.)" if r.get("ambiguous") else "")}
    for f in ("graph", "spatial"):
        row[(f, "gold")] = "yes" if r[f] else "no"
        for name in results:
            row[(f, name)] = cell(results[name]["router"][i][f], r[f])
    rows.append(row)
router_table = pd.DataFrame(rows).set_index("query")
router_table.columns = pd.MultiIndex.from_tuples(router_table.columns)
router_table

graph                    spatial  \
                                              gold vanilla fine-tuned    gold   
query                                                                           
most popular notebooks                         yes  0.07 ✗       0.99      no   
datasets maintained by NOAA                    yes  0.29 ✗       1.00      no   
publications by John Smith                     yes  0.30 ✗       1.00      no   
weather stations near Chicago                   no    0.02       0.00     yes   
Sentinel-2 imagery over Nairobi                 no    0.00       0.00     yes   
NOAA precipitation datasets near Houston       yes  0.00 ✗       0.99     yes   
how do I compute NDVI from Sentinel-2 bands?    no    0.00       0.00      no   
machine learning for flood prediction           no    0.00       0.00      no   
tutorials for GeoPandas                         no    0.04     0.99 ✗      no   

                                                                 
                                             vanilla fine-tuned  
query                                                            
most popular notebooks                          0.10       0.00  
datasets maintained by NOAA                     0.11       0.00  
publications by John Smith                      0.01       0.00  
weather stations near Chicago                 0.28 ✗       1.00  
Sentinel-2 imagery over Nairobi                 0.70       1.00  
NOAA precipitation datasets near Houston      0.25 ✗       1.00  
how do I compute NDVI from Sentinel-2 bands?    0.03       0.00  
machine learning for flood prediction           0.07       0.00  
tutorials for GeoPandas                         0.03       0.00

## First tool: accuracy

A request counts as right when the top choice is one of the tools the gold set accepts.

In [9]:
tool_rows = TOOL_SET[:TOOL_LIMIT]
choice = {name: [max(p, key=p.get) for p in res["tools"]] for name, res in results.items()}
pd.DataFrame({name: {"accuracy": sum(c in r["gold"] for c, r in zip(choice[name], tool_rows)) / len(tool_rows),
                     "requests": len(tool_rows)} for name in results}).T.round(3).astype({"requests": int})

,accuracy,requests
vanilla,0.0,5
fine-tuned,0.8,5


## First tool: every answer

The top choice and its probability; ✓ means it is among the gold tools.

In [10]:
def top(probs, gold):
    best, p = max(probs.items(), key=lambda kv: kv[1])
    return f"{'✓' if best in gold else '✗'} {best} {p:.2f}"


pd.DataFrame([{"request": r["request"][:65],
               "files": "yes" if r["context"] != "Attached files: none" else "no",
               "gold": ", ".join(r["gold"]),
               **{name: top(results[name]["tools"][i], r["gold"]) for name in results}}
              for i, r in enumerate(tool_rows)]).set_index("request")

,files,gold,vanilla,fine-tuned
request,,,,
"Pull the boundary of Travis County, Texas.",no,admin_boundary,✗ add_map_layer 0.48,✓ admin_boundary 1.00
Grid the permits into 1 km squares and shade by count.,yes,aggregate_to_grid,✗ add_map_layer 0.59,✓ aggregate_to_grid 0.99
Look for datasets on wildfire smoke exposure.,no,"request_capability, none",✗ add_map_layer 0.37,✓ request_capability 0.99
What does Moran's I measure?,no,none,✗ add_map_layer 0.65,✓ none 1.00
How alike are central Paris and central Berlin from orbit?,no,"compare_regions, geocode_places",✗ add_map_layer 0.63,✗ none 0.94


## Where the two models disagree

Routing queries where vanilla and fine-tuned Laya reach a different decision on either question, and who matches the gold label.

In [11]:
diff = []
for i, r in enumerate(ROUTER_SET):
    v, f = results["vanilla"]["router"][i], results["fine-tuned"]["router"][i]
    for q in ("graph", "spatial"):
        if (v[q] >= 0.5) != (f[q] >= 0.5):
            diff.append({"query": r["query"][:70], "question": q, "gold": "yes" if r[q] else "no",
                         "vanilla": cell(v[q], r[q]), "fine-tuned": cell(f[q], r[q])})
print(f"{len(diff)} decisions differ out of {2 * len(ROUTER_SET)}")
pd.DataFrame(diff).set_index(["query", "question"])

7 decisions differ out of 18


gold vanilla fine-tuned
query                                    question                        
most popular notebooks                   graph     yes  0.07 ✗       0.99
datasets maintained by NOAA              graph     yes  0.29 ✗       1.00
publications by John Smith               graph     yes  0.30 ✗       1.00
weather stations near Chicago            spatial   yes  0.28 ✗       1.00
NOAA precipitation datasets near Houston graph     yes  0.00 ✗       0.99
                                         spatial   yes  0.25 ✗       1.00
tutorials for GeoPandas                  graph      no    0.04     0.99 ✗

## Your own inputs

Edit the lists and run the cell. Both models are reloaded, which takes about 10–15 s each on CPU.

In [12]:
MY_QUERIES = ["trending flood datasets from NOAA for Louisiana", "notebooks about kriging"]
MY_REQUESTS = [("Make a 500 m buffer around each school.", "schools.geojson (points; columns: name)"),
               ("Which county has the most wells?", "wells.csv (points; columns: id, lat, lon); counties.geojson (polygons; columns: GEOID, NAME)")]

out = []
for name, path in CHECKPOINTS.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = Agent(str(path), device=DEVICE)
    for q in MY_QUERIES:
        a = model.system_one(q, ROUTER_WORDING[name])["answers"]
        out.append({"input": q, "model": name,
                    "answer": f"P(graph) {a['graph']['noul']:.2f} · P(spatial) {a['spatial']['noul']:.2f}"})
    for req, files in MY_REQUESTS:
        p = model.system_one(f"Attached files: {files}\nRequest: {req}", TOOL_Q,
                             max_len=1600, head_max_len=1400)["answers"]["tool"]["probabilities"]
        out.append({"input": req, "model": name,
                    "answer": " · ".join(f"{k} {v:.2f}" for k, v in sorted(p.items(), key=lambda kv: -kv[1])[:3])})
    del model
pd.DataFrame(out).set_index(["input", "model"])

,,answer
input,model,
trending flood datasets from NOAA for Louisiana,vanilla,P(graph) 0.06 · P(spatial) 0.13
notebooks about kriging,vanilla,P(graph) 0.01 · P(spatial) 0.08
Make a 500 m buffer around each school.,vanilla,add_map_layer 0.66 · request_capability 0.11 · count_points_in_are...
Which county has the most wells?,vanilla,add_map_layer 0.46 · cluster_points 0.13 · count_points_in_areas 0.09
trending flood datasets from NOAA for Louisiana,fine-tuned,P(graph) 0.99 · P(spatial) 1.00
notebooks about kriging,fine-tuned,P(graph) 0.00 · P(spatial) 0.00
Make a 500 m buffer around each school.,fine-tuned,buffer_layer 1.00 · execute_code 0.00 · admin_boundary 0.00
Which county has the most wells?,fine-tuned,admin_boundary 0.27 · select_by_attribute 0.10 · vector_spatial_jo...
